In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from getpass import getpass
from time import perf_counter
import hashlib
import json
import os
import random
import re
import unicodedata

from openai import OpenAI

SGD_ROOT = Path("data/sgd")
SPLITS = ("test",)
RESULTS_ROOT = Path("results")
RESULTS_ROOT.mkdir(exist_ok=True)


def load_json(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


schemas_by_split = {}
dialogues_by_split = {}

for split in SPLITS:
    split_root = SGD_ROOT / split

    assert split_root.exists(), f"Cannot find {split_root}"
    assert (split_root / "schema.json").exists()

    schema_list = load_json(split_root / "schema.json")
    schemas_by_split[split] = {
        schema["service_name"]: schema
        for schema in schema_list
    }

    split_dialogues = []
    for path in sorted(split_root.glob("dialogues_*.json")):
        split_dialogues.extend(load_json(path))

    dialogues_by_split[split] = split_dialogues

    print(
        split,
        "| services:",
        len(schemas_by_split[split]),
        "| dialogues:",
        len(split_dialogues),
    )

all_service_names = set().union(
    *(
        set(split_schemas)
        for split_schemas in schemas_by_split.values()
    )
)
print("Candidate services loaded:", len(all_service_names))

In [ ]:
eligible_cases = []

for split in SPLITS:
    split_schemas = schemas_by_split[split]

    for dialogue in dialogues_by_split[split]:
        public_history = []

        for turn_index, turn in enumerate(dialogue["turns"]):
            public_turn = {
                "speaker": turn["speaker"],
                "utterance": turn["utterance"],
            }

            if (
                turn["speaker"] == "USER"
                and len(turn["frames"]) == 1
            ):
                frame = turn["frames"][0]
                state = frame.get("state")
                service = frame["service"]

                intent_schema = None
                if state is not None and service in split_schemas:
                    intent_schema = next(
                        (
                            intent
                            for intent in split_schemas[service]["intents"]
                            if intent["name"] == state["active_intent"]
                        ),
                        None,
                    )

                if (
                    intent_schema is not None
                    and state["active_intent"] != "NONE"
                    and state["slot_values"]
                    and set(intent_schema["required_slots"]).issubset(
                        state["slot_values"]
                    )
                ):
                    case_id = (
                        f'{split}__{dialogue["dialogue_id"]}'
                        f'__turn_{turn_index}'
                    )

                    eligible_cases.append({
                        "case_id": case_id,
                        "dialogue_id": dialogue["dialogue_id"],
                        "dataset_split": split,
                        "service": service,
                        "model_input": {
                            "case_id": case_id,
                            "dataset_split": split,
                            "dialogue_service_count": len(
                                dialogue["services"]
                            ),
                            "dialogue_history": list(public_history),
                            "current_user_utterance": turn["utterance"],
                        },
                        "gold_state": {
                            "service": service,
                            "active_intent": state["active_intent"],
                            "slot_values": state["slot_values"],
                        },
                    })

            public_history.append(public_turn)

print("Eligible user turns:", len(eligible_cases))

In [ ]:
EXPERIMENT_SIZE = 200
EXPERIMENT_SEED = 20260822

assert isinstance(EXPERIMENT_SIZE, int)
assert 1 <= EXPERIMENT_SIZE <= len(eligible_cases)

experiment_cases = random.Random(EXPERIMENT_SEED).sample(
    eligible_cases,
    k=EXPERIMENT_SIZE,
)

assert len({
    case["case_id"]
    for case in experiment_cases
}) == EXPERIMENT_SIZE

experiment_inputs = [
    case["model_input"]
    for case in experiment_cases
]
experiment_gold = {
    case["case_id"]: case["gold_state"]
    for case in experiment_cases
}

EXPERIMENT_ID = (
    f"test_request_state_actionable_size{EXPERIMENT_SIZE}"
    f"_seed{EXPERIMENT_SEED}_unique_datapoints"
)
INPUTS_PATH = (
    RESULTS_ROOT / f"sgd_{EXPERIMENT_ID}_inputs.json"
)
GOLD_PATH = (
    RESULTS_ROOT / f"sgd_{EXPERIMENT_ID}_gold.json"
)

with open(INPUTS_PATH, "w", encoding="utf-8") as file:
    json.dump(
        experiment_inputs,
        file,
        indent=2,
        ensure_ascii=False,
    )

with open(GOLD_PATH, "w", encoding="utf-8") as file:
    json.dump(
        experiment_gold,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Selected datapoints:", len(experiment_cases))
print(
    "Distinct services represented:",
    len({
        case["service"]
        for case in experiment_cases
    }),
)

for case in experiment_cases:
    print(
        case["case_id"],
        "| split:",
        case["dataset_split"],
        "| service:",
        case["service"],
    )

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass(
        "Enter your OpenAI API key: "
    )

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"],
    max_retries=0,
)

MODEL = "gpt-5.6-luna"
EFFORTS = ["none", "low", "medium", "high", "xhigh"]
MAX_OUTPUT_TOKENS = 10000
PROMPT_VERSION = "sgd_request_state_v1"
PRICE_SNAPSHOT_DATE = "2026-08-24"
INPUT_PRICE_PER_MILLION_USD = 0.20
CACHED_INPUT_PRICE_PER_MILLION_USD = 0.02
CACHE_WRITE_PRICE_PER_MILLION_USD = 0.25
OUTPUT_PRICE_PER_MILLION_USD = 1.20

SYSTEM_INSTRUCTIONS = """
You are the request-interpretation stage of an AI customer-service
system. Your output will be used to route the request and prepare
a backend API call.

Given a catalog of available service schemas, the public dialogue
history, and the latest user utterance, recover the exact structured
state of the user's cumulative current request.

Rules:
1. Select exactly one service from the supplied catalog.
2. Select an active_intent defined for that service.
3. Use only slots defined for the selected service.
4. slot_values contains all and only values belonging to the
   cumulative current request, not every fact in the transcript.
5. Return exactly one value for each slot. If the same value was
   expressed in several equivalent ways, return any one wording
   that appeared in the dialogue.
6. Preserve previously specified values that remain part of the
   request and apply the user's latest corrections.
7. Include a system-proposed value only when the user clearly
   accepts it as part of the request.
8. Exclude rejected, superseded, unconfirmed, and system-only
   information.
9. If multiple services appear in the history, choose the service
   associated with the user's current request.
10. Preserve dialogue wording when possible and do not invent
    missing values.
11. Return only the requested structured output.
""".strip()


def build_service_catalog(split):
    catalog = []

    for service_name, schema in schemas_by_split[split].items():
        catalog.append({
            "service": service_name,
            "description": schema["description"],
            "intents": [
                {
                    "name": intent["name"],
                    "description": intent["description"],
                    "required_slots": intent["required_slots"],
                    "optional_slots": intent["optional_slots"],
                }
                for intent in schema["intents"]
            ],
            "slots": schema["slots"],
        })

    return catalog


def build_prompt(case):
    history_text = "\n".join(
        f'{turn["speaker"]}: {turn["utterance"]}'
        for turn in case["dialogue_history"]
    )
    service_catalog = build_service_catalog(
        case["dataset_split"]
    )

    return f"""
AVAILABLE SERVICE CATALOG:
{json.dumps(service_catalog, ensure_ascii=False)}

DIALOGUE HISTORY:
{history_text if history_text else "(No previous turns)"}

CURRENT USER UTTERANCE:
USER: {case["current_user_utterance"]}

Return the service, active intent, and cumulative slot values.
""".strip()


def build_output_format(case):
    split_schemas = schemas_by_split[case["dataset_split"]]
    service_names = sorted(split_schemas)
    intent_names = sorted({
        intent["name"]
        for schema in split_schemas.values()
        for intent in schema["intents"]
    })
    slot_names = sorted({
        slot["name"]
        for schema in split_schemas.values()
        for slot in schema["slots"]
    })

    return {
        "type": "json_schema",
        "name": "sgd_request_state",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "service": {
                    "type": "string",
                    "enum": service_names,
                },
                "active_intent": {
                    "type": "string",
                    "enum": intent_names,
                },
                "slot_values": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "slot": {
                                "type": "string",
                                "enum": slot_names,
                            },
                            "values": {
                                "type": "array",
                                "items": {"type": "string"},
                                "minItems": 1,
                                "maxItems": 1,
                            },
                        },
                        "required": ["slot", "values"],
                        "additionalProperties": False,
                    },
                    "minItems": 1,
                },
            },
            "required": [
                "service",
                "active_intent",
                "slot_values",
            ],
            "additionalProperties": False,
        },
    }

In [ ]:
SCORING_VERSION = "sgd_strict_exact_v1"


def normalize_value(value):
    value = unicodedata.normalize("NFKC", str(value))
    value = value.casefold().replace("’", "'")
    value = re.sub(r"\s+", " ", value)
    return value.strip()


def convert_prediction_slots(prediction):
    slot_dict = {}
    duplicate_slots = set()

    for item in prediction["slot_values"]:
        slot = item["slot"]
        if slot in slot_dict:
            duplicate_slots.add(slot)
        else:
            slot_dict[slot] = item["values"]

    return slot_dict, duplicate_slots


def score_sgd_prediction(prediction, gold_state, split):
    """Score complete request-state recovery using strict matching only."""
    predicted_slots, duplicate_slots = convert_prediction_slots(prediction)
    gold_slots = gold_state["slot_values"]
    split_schemas = schemas_by_split[split]

    same_slot_names = (
        set(predicted_slots) == set(gold_slots)
        and not duplicate_slots
    )
    individual_slot_match_details = {}
    for slot in sorted(set(predicted_slots) | set(gold_slots)):
        predicted_values = predicted_slots.get(slot)
        gold_values = gold_slots.get(slot)
        comparable = (
            slot in predicted_slots
            and slot in gold_slots
            and slot not in duplicate_slots
            and len(predicted_values) == 1
        )
        strict_matches = (
            [
                value for value in gold_values
                if normalize_value(predicted_values[0])
                == normalize_value(value)
            ]
            if comparable else []
        )
        individual_slot_match_details[slot] = {
            "strict_correct": bool(strict_matches),
            "matched_gold_value": (
                strict_matches[0] if strict_matches else None
            ),
            "predicted_values": predicted_values,
            "gold_values": gold_values,
        }

    individual_slot_results_strict = {
        slot: details["strict_correct"]
        for slot, details in individual_slot_match_details.items()
    }
    slot_values_strict_exact = (
        same_slot_names
        and all(individual_slot_results_strict[slot] for slot in gold_slots)
    )

    service_correct = prediction["service"] == gold_state["service"]
    intent_correct = (
        prediction["active_intent"] == gold_state["active_intent"]
    )
    predicted_service_schema = split_schemas.get(prediction["service"])
    service_intent_valid = (
        predicted_service_schema is not None
        and prediction["active_intent"] in {
            intent["name"]
            for intent in predicted_service_schema["intents"]
        }
    )
    predicted_service_slots = (
        {slot["name"] for slot in predicted_service_schema["slots"]}
        if predicted_service_schema is not None else set()
    )
    slot_names_valid = set(predicted_slots).issubset(predicted_service_slots)
    resolved_strict_exact = (
        service_correct
        and intent_correct
        and slot_values_strict_exact
        and service_intent_valid
        and slot_names_valid
    )

    return {
        "service_correct": int(service_correct),
        "intent_correct": int(intent_correct),
        "service_intent_valid": int(service_intent_valid),
        "slot_names_valid": int(slot_names_valid),
        "individual_slot_results_strict": individual_slot_results_strict,
        "individual_slot_match_details": individual_slot_match_details,
        "slot_values_strict_exact": int(slot_values_strict_exact),
        "resolved_strict_exact": int(resolved_strict_exact),
    }

In [ ]:
def estimate_cost_usd(
    input_tokens,
    cached_input_tokens,
    cache_write_tokens,
    output_tokens,
):
    uncached_input_tokens = max(
        input_tokens
        - cached_input_tokens
        - cache_write_tokens,
        0,
    )
    cost = (
        uncached_input_tokens * INPUT_PRICE_PER_MILLION_USD
        + cached_input_tokens
        * CACHED_INPUT_PRICE_PER_MILLION_USD
        + cache_write_tokens
        * CACHE_WRITE_PRICE_PER_MILLION_USD
        + output_tokens * OUTPUT_PRICE_PER_MILLION_USD
    ) / 1_000_000
    return uncached_input_tokens, cost


def run_sgd_case(case, gold_state, effort, repetition=1):
    if effort not in EFFORTS:
        raise ValueError(
            f"Unknown effort {effort!r}. Expected one of {EFFORTS}."
        )

    service_schema = schemas_by_split[
        case["dataset_split"]
    ][gold_state["service"]]
    intent_schema = next(
        intent
        for intent in service_schema["intents"]
        if intent["name"] == gold_state["active_intent"]
    )
    utterances = [
        turn["utterance"]
        for turn in case["dialogue_history"]
    ] + [case["current_user_utterance"]]
    dialogue_text = " ".join(utterances)

    prompt = build_prompt(case)
    prompt_hash = hashlib.sha256(
        (SYSTEM_INSTRUCTIONS + "\n\n" + prompt).encode("utf-8")
    ).hexdigest()

    record = {
        "case_id": case["case_id"],
        "service": gold_state["service"],
        "gold_service": gold_state["service"],
        "active_intent": gold_state["active_intent"],
        "gold_active_intent": gold_state["active_intent"],
        "dataset_split": case["dataset_split"],
        "reasoning_effort": effort,
        "repetition": repetition,
        "history_turn_count": len(case["dialogue_history"]),
        "dialogue_turn_count": len(utterances),
        "dialogue_word_count": len(
            re.findall(r"\S+", dialogue_text)
        ),
        "dialogue_character_count": len(dialogue_text),
        "dialogue_service_count": case[
            "dialogue_service_count"
        ],
        "is_multi_domain": int(
            case["dialogue_service_count"] > 1
        ),
        "gold_slot_count": len(gold_state["slot_values"]),
        "required_slot_count": len(
            intent_schema["required_slots"]
        ),
        "intent_is_transactional": int(
            intent_schema["is_transactional"]
        ),
        "requested_model": MODEL,
        "prompt_version": PROMPT_VERSION,
        "scoring_version": SCORING_VERSION,
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "prompt_sha256": prompt_hash,
        "success": False,
        "latency_seconds": None,
        "input_tokens": None,
        "cached_input_tokens": None,
        "cache_write_tokens": None,
        "uncached_input_tokens": None,
        "output_tokens": None,
        "reasoning_tokens": None,
        "visible_output_tokens": None,
        "total_tokens": None,
        "estimated_cost_usd": None,
        "pricing": {
            "snapshot_date": PRICE_SNAPSHOT_DATE,
            "input_per_million_usd": (
                INPUT_PRICE_PER_MILLION_USD
            ),
            "cached_input_per_million_usd": (
                CACHED_INPUT_PRICE_PER_MILLION_USD
            ),
            "cache_write_per_million_usd": (
                CACHE_WRITE_PRICE_PER_MILLION_USD
            ),
            "output_per_million_usd": (
                OUTPUT_PRICE_PER_MILLION_USD
            ),
        },
        "predicted_service": None,
        "predicted_active_intent": None,
        "predicted_slot_count": None,
        "service_correct": None,
        "intent_correct": None,
        "slot_values_strict_exact": None,
        "resolved_strict_exact": None,
        "error_type": None,
        "error_message": None,
    }

    start_time = perf_counter()

    try:
        response = client.responses.create(
            model=MODEL,
            instructions=SYSTEM_INSTRUCTIONS,
            input=prompt,
            reasoning={
                "effort": effort,
                "context": "current_turn",
            },
            text={
                "verbosity": "low",
                "format": build_output_format(case),
            },
            max_output_tokens=MAX_OUTPUT_TOKENS,
            service_tier="default",
            store=False,
            prompt_cache_options={"mode": "explicit"},
        )
        record["latency_seconds"] = (
            perf_counter() - start_time
        )
    except Exception as error:
        record["latency_seconds"] = (
            perf_counter() - start_time
        )
        record["error_type"] = type(error).__name__
        record["error_message"] = str(error)
        return record

    record.update({
        "response_id": response.id,
        "returned_model": response.model,
        "response_status": response.status,
        "service_tier": getattr(
            response, "service_tier", None
        ),
    })

    if response.usage is not None:
        usage = response.usage
        input_details = getattr(
            usage, "input_tokens_details", None
        )
        output_details = getattr(
            usage, "output_tokens_details", None
        )

        cached_input_tokens = (
            getattr(input_details, "cached_tokens", 0) or 0
        )
        cache_write_tokens = (
            getattr(input_details, "cache_write_tokens", 0)
            or 0
        )
        reasoning_tokens = (
            getattr(output_details, "reasoning_tokens", 0)
            or 0
        )
        uncached_input_tokens, estimated_cost_usd = (
            estimate_cost_usd(
                usage.input_tokens,
                cached_input_tokens,
                cache_write_tokens,
                usage.output_tokens,
            )
        )

        record.update({
            "input_tokens": usage.input_tokens,
            "cached_input_tokens": cached_input_tokens,
            "cache_write_tokens": cache_write_tokens,
            "uncached_input_tokens": uncached_input_tokens,
            "output_tokens": usage.output_tokens,
            "reasoning_tokens": reasoning_tokens,
            "visible_output_tokens": max(
                usage.output_tokens - reasoning_tokens,
                0,
            ),
            "total_tokens": usage.total_tokens,
            "estimated_cost_usd": estimated_cost_usd,
        })

    if response.status != "completed":
        record["error_type"] = "IncompleteResponse"
        record["error_message"] = str(
            response.incomplete_details
        )
        return record

    try:
        prediction = json.loads(response.output_text)
        scores = score_sgd_prediction(
            prediction,
            gold_state,
            case["dataset_split"],
        )
    except Exception as error:
        record["error_type"] = type(error).__name__
        record["error_message"] = str(error)
        record["raw_output"] = response.output_text
        return record

    record.update({
        "success": True,
        "predicted_service": prediction["service"],
        "predicted_active_intent": prediction[
            "active_intent"
        ],
        "predicted_slot_count": len(
            prediction["slot_values"]
        ),
        "service_correct": scores["service_correct"],
        "intent_correct": scores["intent_correct"],
        "slot_values_strict_exact": scores[
            "slot_values_strict_exact"
        ],
        "resolved_strict_exact": scores[
            "resolved_strict_exact"
        ],
        "prediction": prediction,
        "gold_state": gold_state,
        "scores": scores,
    })
    return record


def append_result_jsonl(result, path):
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "a", encoding="utf-8") as file:
        file.write(
            json.dumps(result, ensure_ascii=False) + "\n"
        )

In [ ]:
EXPERIMENT_RESULTS_PATH = (
    RESULTS_ROOT
    / f"sgd_{EXPERIMENT_ID}_{PROMPT_VERSION}_runs.jsonl"
)
RUN_PAID_EXPERIMENT = False
MAX_NEW_CALLS = None  # Keep 1 for the smoke test; then use None.

assert MAX_NEW_CALLS is None or (
    isinstance(MAX_NEW_CALLS, int)
    and MAX_NEW_CALLS >= 1
)

case_by_id = {
    case["case_id"]: case
    for case in experiment_inputs
}
assert len(case_by_id) == EXPERIMENT_SIZE
assert set(case_by_id) == set(experiment_gold)

for case_id, case in case_by_id.items():
    gold = experiment_gold[case_id]
    assert case["dataset_split"] == "test"
    assert "service" not in case
    assert "service_schema" not in case
    assert case["dialogue_service_count"] >= 1
    assert gold["service"] not in case_id

    service_schema = schemas_by_split["test"][
        gold["service"]
    ]
    intent_schema = next(
        intent
        for intent in service_schema["intents"]
        if intent["name"] == gold["active_intent"]
    )
    required_slots = set(intent_schema["required_slots"])
    allowed_slots = {
        slot["name"]
        for slot in service_schema["slots"]
    }
    assert required_slots.issubset(gold["slot_values"])
    assert set(gold["slot_values"]).issubset(allowed_slots)
    assert all(gold["slot_values"].values())

schedule = [
    {
        "case_id": case["case_id"],
        "reasoning_effort": effort,
        "repetition": 1,
        "prompt_version": PROMPT_VERSION,
    }
    for case in experiment_inputs
    for effort in EFFORTS
]

random.Random(EXPERIMENT_SEED).shuffle(schedule)
TOTAL_CALLS = len(schedule)

for position, job in enumerate(schedule, start=1):
    job["schedule_position"] = position


def job_key(record):
    return (
        record["case_id"],
        record["reasoning_effort"],
        record.get("repetition", 1),
        record.get("prompt_version"),
    )


scheduled_keys = {job_key(job) for job in schedule}
assert len(scheduled_keys) == TOTAL_CALLS

expected_prompt_hashes = {
    case_id: hashlib.sha256(
        (
            SYSTEM_INSTRUCTIONS
            + "\n\n"
            + build_prompt(case)
        ).encode("utf-8")
    ).hexdigest()
    for case_id, case in case_by_id.items()
}


existing_results = []
if EXPERIMENT_RESULTS_PATH.exists():
    with open(
        EXPERIMENT_RESULTS_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        existing_results = [
            json.loads(line)
            for line in file
            if line.strip()
        ]

for result in existing_results:
    assert job_key(result) in scheduled_keys
    assert result["requested_model"] == MODEL
    assert result["prompt_sha256"] == (
        expected_prompt_hashes[result["case_id"]]
    )

successful_results = [
    result
    for result in existing_results
    if result.get("success") is True
]
recorded_keys = {
    job_key(result)
    for result in successful_results
}
assert len(recorded_keys) == len(successful_results)

failed_attempts = len(existing_results) - len(successful_results)
remaining_calls = TOTAL_CALLS - len(recorded_keys)

print("Experiment size:", EXPERIMENT_SIZE)
print("Total successful calls required:", TOTAL_CALLS)
print("Successful calls recorded:", len(recorded_keys))
print("Failed attempts recorded:", failed_attempts)
print("Calls remaining:", remaining_calls)
print("Results path:", EXPERIMENT_RESULTS_PATH)

if remaining_calls and not RUN_PAID_EXPERIMENT:
    raise RuntimeError(
        "Paid calls are disabled. Review the information above, "
        "then set RUN_PAID_EXPERIMENT = True."
    )

new_calls = 0

for job in schedule:
    if job_key(job) in recorded_keys:
        continue

    case = case_by_id[job["case_id"]]
    print(
        f'[{job["schedule_position"]:02d}/{TOTAL_CALLS}] '
        f'{job["case_id"]} | '
        f'effort={job["reasoning_effort"]}'
    )

    result = run_sgd_case(
        case=case,
        gold_state=experiment_gold[job["case_id"]],
        effort=job["reasoning_effort"],
        repetition=job["repetition"],
    )

    result.update({
        "attempt_id": (
            f'{job["case_id"]}__'
            f'{job["reasoning_effort"]}__'
            f'r{job["repetition"]}'
        ),
        "experiment_size": EXPERIMENT_SIZE,
        "experiment_seed": EXPERIMENT_SEED,
        "schedule_position": job["schedule_position"],
        "reasoning_context": "current_turn",
        "requested_service_tier": "default",
        "max_output_tokens": MAX_OUTPUT_TOKENS,
        "prompt_cache_mode": "explicit",
    })

    append_result_jsonl(
        result,
        EXPERIMENT_RESULTS_PATH,
    )

    if result["success"]:
        recorded_keys.add(job_key(job))
        new_calls += 1
        print(
            f'  latency={result["latency_seconds"]:.3f}s | '
            f'resolved_strict_exact={result["resolved_strict_exact"]} | '
            f'input_tokens={result.get("input_tokens")} | '
            f'output_tokens={result.get("output_tokens")} | '
            f'reasoning_tokens={result.get("reasoning_tokens")} | '
            f'estimated_cost_usd='
            f'{result.get("estimated_cost_usd")}'
        )
    else:
        print(
            f'  failed | {result["error_type"]}: '
            f'{result["error_message"]}'
        )
        raise RuntimeError(
            "Experiment stopped after the first failed call. "
            "No later jobs were submitted."
        )

    if (
        MAX_NEW_CALLS is not None
        and new_calls >= MAX_NEW_CALLS
    ):
        print(
            "Paused at MAX_NEW_CALLS. Inspect the smoke-test "
            "record before continuing."
        )
        break

print("Successful calls recorded:", len(recorded_keys))
print("Saved to:", EXPERIMENT_RESULTS_PATH)